In [1]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server']


In [2]:
# Your connection parameters
server = 'DD-HIEUHC'
db_name = 'master'
database = 'BicycleRetailer'

# Establish the connection
conn = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};'
                      'SERVER=' + server + ';'
                      'DATABASE=' + database + ';'
                      'Trusted_Connection=yes;'
                      'Encrypt=yes;'
                      'TrustServerCertificate=yes; ')

In [3]:
import pandas as pd 
Query = """ select * from Production.Culture """
df = pd.read_sql(Query, conn)

C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel_6936\542319313.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(Query, conn)


In [4]:
df

,CultureID,Name,ModifiedDate
0,,Invariant Language (Invariant Country),2008-04-30
1,ar,Arabic,2008-04-30
2,en,English,2008-04-30
3,es,Spanish,2008-04-30
4,fr,French,2008-04-30
5,he,Hebrew,2008-04-30
6,th,Thai,2008-04-30
7,zh-cht,Chinese,2008-04-30


In [6]:
import snowflake.connector 

In [7]:
import snowflake.connector

# Thông tin kết nối
conn = snowflake.connector.connect(
    user='Huynhhieu1710',
    password='17102003Huynhconghieu@',
    account='ij46654.southeast-asia.azure',  
    warehouse='DATA_WAREHOUSE_DEMO',
    database='PERSON_TRANS_COUNTRY_DTB',
    schema='PUBLIC',
    role='SYSADMIN'
)
cursor = conn.cursor()
cursor.execute("SELECT * FROM COUNTRY_BUSINESS LIMIT 10")
rows = cursor.fetchall()
print("Connect succesfull with Snowflake!")


Connect succesfull with Snowflake!


In [8]:
Query = """ select * from COUNTRY_BUSINESS """
df = pd.read_sql(Query, conn)

C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel_6936\551395944.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(Query, conn)


In [9]:
df

,BUSINESSENTITYID,STATEPROVINCECODE,COUNTRYREGIONCODE,STATEPROVINCENAME,COUNTRYREGIONNAME
0,16496,NSW,AU,New South Wales,Australia
1,12506,NSW,AU,New South Wales,Australia
2,11390,NSW,AU,New South Wales,Australia
3,10798,BC,CA,British Columbia,Canada
4,12283,93,FR,Seine Saint Denis,France
...,...,...,...,...,...
18793,19884,59,FR,Nord,France
18794,15339,CA,US,California,United States
18795,15308,SL,DE,Saarland,Germany
18796,3917,WA,US,Washington,United States


In [10]:
import pyodbc
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

In [11]:
def generate_create_table_sql(df, table_name):
    dtype_mapping = {
        "int64": "NUMBER",
        "float64": "FLOAT",
        "object": "STRING",
        "bool": "BOOLEAN",
        "datetime64[ns]": "TIMESTAMP_NTZ"
    }

    columns = []
    for col, dtype in df.dtypes.items():
        snow_type = dtype_mapping.get(str(dtype), "STRING")
        columns.append(f'"{col}" {snow_type}')
    
    columns_sql = ",\n  ".join(columns)
    return f'CREATE OR REPLACE TABLE "{table_name.upper()}" (\n  {columns_sql}\n);'

In [12]:
server = 'DD-HIEUHC'
db_name = 'master'
database = 'BicycleRetailer'
def get_data_from_sqlserver(server, database, table_name):
    conn_str = (
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Trusted_Connection=yes;"
        f"Encrypt=yes;"
        f"TrustServerCertificate=yes;"
    )
    conn = pyodbc.connect(conn_str)
    query = f"SELECT * FROM {table_name}"
    df = pd.read_sql(query, conn)
    conn.close()
    print(f"Đã lấy {len(df)} dòng từ bảng {table_name} (SQL Server)")
    return df

In [13]:
conn = snowflake.connector.connect(
    user='Huynhhieu1710',
    password='17102003Huynhconghieu@',
    account='ij46654.southeast-asia.azure',  
    warehouse='DATA_WAREHOUSE_DEMO',
    database='PERSON_TRANS_COUNTRY_DTB',
    schema='PUBLIC',
    role='SYSADMIN'
)
cursor = conn.cursor()
cursor.execute("SELECT * FROM COUNTRY_BUSINESS LIMIT 10")
rows = cursor.fetchall()
print("Connect succesfull with Snowflake!")

Connect succesfull with Snowflake!


In [14]:
def push_df_to_snowflake(df, table_name, conn_sf):
    cursor = conn_sf.cursor()

    # Tạo bảng tự động
    create_sql = generate_create_table_sql(df, table_name)
    cursor.execute(create_sql)
    print(f"Đã tạo bảng {table_name} trên Snowflake")

    # Ghi dữ liệu
    success, nchunks, nrows, _ = write_pandas(conn_sf, df, table_name=table_name.upper())
    print(f"Đã đẩy {nrows} dòng vào bảng {table_name} trên Snowflake")

    cursor.close()

## Test flow 

In [ ]:
import pyodbc
import snowflake.connector
import pandas as pd
from sqlalchemy import create_engine, text
import urllib.parse
from datetime import datetime
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Đã import thành công tất cả thư viện cần thiết!")

Đã import thành công tất cả thư viện cần thiết!


In [16]:
def connect_sql_server():
    """Kết nối với SQL Server"""
    try:
        server = 'DD-HIEUHC'
        database = 'BicycleRetailer'
        
        conn = pyodbc.connect(
            'DRIVER={ODBC Driver 17 for SQL Server};'
            'SERVER=' + server + ';'
            'DATABASE=' + database + ';'
            'Trusted_Connection=yes;'
            'Encrypt=yes;'
            'TrustServerCertificate=yes;'
        )
        
        logger.info("Kết nối SQL Server thành công!")
        return conn
    except Exception as e:
        logger.error(f"Lỗi kết nối SQL Server: {e}")
        return None

In [17]:
connect_sql_server()

2025-07-17 14:05:07,515 - INFO - Kết nối SQL Server thành công!


In [18]:
def connect_snowflake():
    """Kết nối với Snowflake"""
    try:
        conn = snowflake.connector.connect(
            user='Huynhhieu1710',
            password='17102003Huynhconghieu@',
            account='ij46654.southeast-asia.azure',
            warehouse='DATA_WAREHOUSE_DEMO',
            database='PERSON_TRANS_COUNTRY_DTB',
            schema='PUBLIC',
            role='SYSADMIN'
        )
        
        logger.info("Kết nối Snowflake thành công!")
        return conn
    except Exception as e:
        logger.error(f"Lỗi kết nối Snowflake: {e}")
        return None

In [19]:
connect_snowflake()

2025-07-17 14:05:07,535 - INFO - Snowflake Connector for Python Version: 3.16.0, Python Version: 3.12.8, Platform: Windows-11-10.0.26100-SP0
2025-07-17 14:05:07,536 - INFO - Connecting to GLOBAL Snowflake domain
2025-07-17 14:05:07,982 - INFO - Kết nối Snowflake thành công!


### Quét database để check schema and table


In [20]:
def get_sql_server_tables():
    """Lấy danh sách tất cả bảng từ SQL Server"""
    conn = connect_sql_server()
    if not conn:
        return []
    
    try:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT TABLE_SCHEMA, TABLE_NAME 
            FROM INFORMATION_SCHEMA.TABLES 
            WHERE TABLE_TYPE = 'BASE TABLE'
            ORDER BY TABLE_SCHEMA, TABLE_NAME
        """)
        
        tables = []
        for row in cursor.fetchall():
            tables.append({
                'schema': row[0],
                'table': row[1],
                'full_name': f"{row[0]}.{row[1]}"
            })
        
        cursor.close()
        conn.close()
        
        logger.info(f"Tìm thấy {len(tables)} bảng trong SQL Server")
        return tables
        
    except Exception as e:
        logger.error(f"Lỗi khi lấy danh sách bảng: {e}")
        conn.close()
        return []

In [21]:
get_sql_server_tables()

2025-07-17 14:05:08,015 - INFO - Kết nối SQL Server thành công!
2025-07-17 14:05:08,027 - INFO - Tìm thấy 51 bảng trong SQL Server


[{'schema': 'dbo',
  'table': 'AWBuildVersion',
  'full_name': 'dbo.AWBuildVersion'},
 {'schema': 'dbo', 'table': 'DatabaseLog', 'full_name': 'dbo.DatabaseLog'},
 {'schema': 'dbo', 'table': 'ErrorLog', 'full_name': 'dbo.ErrorLog'},
 {'schema': 'dbo', 'table': 'sysdiagrams', 'full_name': 'dbo.sysdiagrams'},
 {'schema': 'Production',
  'table': 'BillOfMaterials',
  'full_name': 'Production.BillOfMaterials'},
 {'schema': 'Production',
  'table': 'Culture',
  'full_name': 'Production.Culture'},
 {'schema': 'Production',
  'table': 'Document',
  'full_name': 'Production.Document'},
 {'schema': 'Production',
  'table': 'Illustration',
  'full_name': 'Production.Illustration'},
 {'schema': 'Production',
  'table': 'Location',
  'full_name': 'Production.Location'},
 {'schema': 'Production',
  'table': 'Product',
  'full_name': 'Production.Product'},
 {'schema': 'Production',
  'table': 'ProductCategory',
  'full_name': 'Production.ProductCategory'},
 {'schema': 'Production',
  'table': 'Produc

#### Đếm số bảng và liệt kê trong 1 database

In [22]:
tables = get_sql_server_tables()

print("📋 Danh sách bảng trong SQL Server:")
print("-" * 50)
for i, table in enumerate(tables, 1):
    print(f"{i:2d}. {table['full_name']}")

# Lưu danh sách bảng để sử dụng sau
print(f"\n✅ Tổng cộng có {len(tables)} bảng")

2025-07-17 14:05:08,043 - INFO - Kết nối SQL Server thành công!
2025-07-17 14:05:08,051 - INFO - Tìm thấy 51 bảng trong SQL Server


📋 Danh sách bảng trong SQL Server:
--------------------------------------------------
 1. dbo.AWBuildVersion
 2. dbo.DatabaseLog
 3. dbo.ErrorLog
 4. dbo.sysdiagrams
 5. Production.BillOfMaterials
 6. Production.Culture
 7. Production.Document
 8. Production.Illustration
 9. Production.Location
10. Production.Product
11. Production.ProductCategory
12. Production.ProductCostHistory
13. Production.ProductDescription
14. Production.ProductDocument
15. Production.ProductInventory
16. Production.ProductListPriceHistory
17. Production.ProductModel
18. Production.ProductModelIllustration
19. Production.ProductModelProductDescriptionCulture
20. Production.ProductPhoto
21. Production.ProductProductPhoto
22. Production.ProductReview
23. Production.ProductSubcategory
24. Production.ScrapReason
25. Production.UnitMeasure
26. Production.WorkOrder
27. Production.WorkOrderRouting
28. Purchasing.ProductVendor
29. Purchasing.PurchaseOrderDetail
30. Purchasing.PurchaseOrderHeader
31. Purchasing.ShipMeth

In [23]:
def read_sql_server_data(table_name, limit=None):
    """Đọc dữ liệu từ bảng SQL Server"""
    conn = connect_sql_server()
    if not conn:
        return None
    
    try:
        # Tạo query với hoặc không có limit
        if limit:
            query = f"SELECT TOP {limit} * FROM {table_name}"
        else:
            query = f"SELECT * FROM {table_name}"
        
        # Sử dụng pandas để đọc dữ liệu
        df = pd.read_sql(query, conn)
        
        conn.close()
        logger.info(f"Đọc thành công {len(df)} rows từ bảng {table_name}")
        return df
        
    except Exception as e:
        logger.error(f"Lỗi khi đọc dữ liệu từ {table_name}: {e}")
        conn.close()
        return None

In [24]:
read_sql_server_data("Production.Culture", limit=None)

2025-07-17 14:05:08,071 - INFO - Kết nối SQL Server thành công!
C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel_6936\3744377393.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
2025-07-17 14:05:08,076 - INFO - Đọc thành công 8 rows từ bảng Production.Culture


,CultureID,Name,ModifiedDate
0,,Invariant Language (Invariant Country),2008-04-30
1,ar,Arabic,2008-04-30
2,en,English,2008-04-30
3,es,Spanish,2008-04-30
4,fr,French,2008-04-30
5,he,Hebrew,2008-04-30
6,th,Thai,2008-04-30
7,zh-cht,Chinese,2008-04-30


#### Đọc dữ liệu cả bảng

In [25]:
if tables:
    # Lấy bảng bất kỳ để test
    test_table = tables[50]['full_name'] #Điều chính thứ tự bảng muốn đọc trong database
    print(f"🔍 Test đọc dữ liệu từ bảng: {test_table}")
    
    # Đọc 10 rows đầu tiên để test
    sample_data = read_sql_server_data(test_table, limit=10)
    
    if sample_data is not None:
        print(f"Đọc thành công! Shape: {sample_data.shape}")
        print("\nThông tin về bảng:")
        print(sample_data.info())
        print("\n5 rows đầu tiên:")
        print(sample_data.head())
    else:
        print("Không thể đọc dữ liệu từ bảng này")
else:
    print("Không có bảng nào để test")

2025-07-17 14:05:08,089 - INFO - Kết nối SQL Server thành công!
C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel_6936\3744377393.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
2025-07-17 14:05:08,094 - INFO - Đọc thành công 10 rows từ bảng Sales.Store


🔍 Test đọc dữ liệu từ bảng: Sales.Store
Đọc thành công! Shape: (10, 6)

Thông tin về bảng:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   BusinessEntityID  10 non-null     int64         
 1   Name              10 non-null     object        
 2   SalesPersonID     10 non-null     int64         
 3   Demographics      10 non-null     object        
 4   rowguid           10 non-null     object        
 5   ModifiedDate      10 non-null     datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 612.0+ bytes
None

5 rows đầu tiên:
   BusinessEntityID                            Name  SalesPersonID  \
0               292            Next-Door Bike Store            279   
1               294  Professional Sales and Service            276   
2               296                  Riders Company            277   

## Tạo bảng trong Snowflake dựa trên DataFrame

##### Cấp quyền admin

In [14]:
## xuát hiện lỗi ko thể tạo bảng mặc dù user là admin, nhưng chưa cấp quyền thì ko create được

def grant_permissions():
    """Cấp quyền CREATE TABLE thông qua Python"""
    conn = connect_snowflake()
    if not conn:
        return False
    
    try:
        cursor = conn.cursor()
        
        # Chuyển sang role ACCOUNTADMIN
        cursor.execute("USE ROLE ACCOUNTADMIN")
        
        # Cấp quyền
        permissions = [
            "GRANT CREATE TABLE ON SCHEMA PUBLIC TO ROLE SYSADMIN",
            "GRANT USAGE ON SCHEMA PUBLIC TO ROLE SYSADMIN",
            "GRANT USAGE ON DATABASE PERSON_TRANS_COUNTRY_DTB TO ROLE SYSADMIN",
            "GRANT CREATE SCHEMA ON DATABASE PERSON_TRANS_COUNTRY_DTB TO ROLE SYSADMIN",
            "GRANT ROLE SYSADMIN TO USER HUYNHHIEU1710"
        ]
        
        for perm in permissions:
            try:
                cursor.execute(perm)
                print(f"✓ Thành công: {perm}")
            except Exception as e:
                print(f"✗ Lỗi: {perm} - {e}")
        
        # Kiểm tra quyền
        cursor.execute("SHOW GRANTS TO ROLE SYSADMIN")
        grants = cursor.fetchall()
        
        print("\n=== QUYỀN CỦA ROLE SYSADMIN ===")
        for grant in grants:
            print(f"- {grant}")
        
        cursor.close()
        conn.close()
        return True
        
    except Exception as e:
        print(f"Lỗi khi cấp quyền: {e}")
        if conn:
            conn.close()
        return False
grant_permissions()

✓ Thành công: GRANT CREATE TABLE ON SCHEMA PUBLIC TO ROLE SYSADMIN
✓ Thành công: GRANT USAGE ON SCHEMA PUBLIC TO ROLE SYSADMIN
✓ Thành công: GRANT USAGE ON DATABASE PERSON_TRANS_COUNTRY_DTB TO ROLE SYSADMIN
✓ Thành công: GRANT CREATE SCHEMA ON DATABASE PERSON_TRANS_COUNTRY_DTB TO ROLE SYSADMIN
✓ Thành công: GRANT ROLE SYSADMIN TO USER HUYNHHIEU1710

=== QUYỀN CỦA ROLE SYSADMIN ===
- (datetime.datetime(2025, 7, 14, 10, 30, 14, 552000, tzinfo=<DstTzInfo 'Asia/Ho_Chi_Minh' +07+7:00:00 STD>), 'CREATE DATABASE', 'ACCOUNT', 'IJ46654', 'ROLE', 'SYSADMIN', 'true', '')
- (datetime.datetime(2025, 7, 14, 10, 30, 14, 553000, tzinfo=<DstTzInfo 'Asia/Ho_Chi_Minh' +07+7:00:00 STD>), 'CREATE WAREHOUSE', 'ACCOUNT', 'IJ46654', 'ROLE', 'SYSADMIN', 'true', '')
- (datetime.datetime(2025, 7, 15, 14, 30, 22, 357000, tzinfo=<DstTzInfo 'Asia/Ho_Chi_Minh' +07+7:00:00 STD>), 'CREATE SCHEMA', 'DATABASE', 'PERSON_TRANS_COUNTRY_DTB', 'ROLE', 'SYSADMIN', 'false', 'ACCOUNTADMIN')
- (datetime.datetime(2025, 7, 15, 14,

True

## Create table in Snowflake

In [27]:
def create_snowflake_table(df, table_name):
    """Tạo bảng trong Snowflake dựa trên DataFrame"""
    conn = connect_snowflake()
    if not conn:
        return False
    
    try:
        cursor = conn.cursor()        
        # Thiết lập role và warehouse
        cursor.execute("USE ROLE SYSADMIN")
        cursor.execute("USE WAREHOUSE DATA_WAREHOUSE_DEMO")  
        cursor.execute("USE DATABASE PERSON_TRANS_COUNTRY_DTB")
        
        # Thử các schema khác nhau
        
        schemas_to_try = ["DBO", "PRODUCTION", "SALES", "PURCHASING", "PUBLIC"]
        
        # Tạo CREATE TABLE statement
        columns = []
        for col, dtype in df.dtypes.items():
            if dtype == 'object':
                col_type = 'VARCHAR(500)'
            elif dtype == 'int64':
                col_type = 'INTEGER'
            elif dtype == 'float64':
                col_type = 'FLOAT'
            elif dtype == 'datetime64[ns]':
                col_type = 'TIMESTAMP'
            elif dtype == 'bool':
                col_type = 'BOOLEAN'
            else:
                col_type = 'VARCHAR(500)'
            
            columns.append(f"{col} {col_type}")
        
        create_sql = f"""
        CREATE OR REPLACE TABLE {table_name} (
            {', '.join(columns)}
        )
        """
        
        cursor.execute(create_sql)
        cursor.close()
        conn.close()
        
        logger.info(f"Tạo bảng {table_name} thành công trong Snowflake")
        return True
        
    except Exception as e:
        logger.error(f"Lỗi khi tạo bảng {table_name}: {e}")
        conn.close()
        return False

In [28]:
create_snowflake_table(df, "Production.Culture")

2025-07-17 14:05:08,133 - INFO - Snowflake Connector for Python Version: 3.16.0, Python Version: 3.12.8, Platform: Windows-11-10.0.26100-SP0
2025-07-17 14:05:08,134 - INFO - Connecting to GLOBAL Snowflake domain
2025-07-17 14:05:08,410 - INFO - Kết nối Snowflake thành công!
2025-07-17 14:05:09,085 - INFO - Tạo bảng Production.Culture thành công trong Snowflake


True

##### Kiểm tra quyền trong snowflake

In [29]:
def check_current_grants():
    """Kiểm tra quyền hiện tại của role"""
    conn = connect_snowflake()
    if not conn:
        return False
    
    try:
        cursor = conn.cursor()
        
        # Kiểm tra quyền của role hiện tại
        cursor.execute("SHOW GRANTS TO ROLE SYSADMIN")
        grants = cursor.fetchall()
        
        print("=== QUYỀN CỦA ROLE SYSADMIN ===")
        for grant in grants:
            print(f"- {grant}")
        
        # Kiểm tra quyền trên schema PUBLIC
        cursor.execute("SHOW GRANTS ON SCHEMA PUBLIC")
        schema_grants = cursor.fetchall()
        
        print("\n=== QUYỀN TRÊN SCHEMA PUBLIC ===")
        for grant in schema_grants:
            print(f"- {grant}")
        
        cursor.close()
        conn.close()
        return True
        
    except Exception as e:
        print(f"Lỗi khi kiểm tra quyền: {e}")
        if conn:
            conn.close()
        return False

# Chạy kiểm tra
check_current_grants()

2025-07-17 14:05:09,100 - INFO - Snowflake Connector for Python Version: 3.16.0, Python Version: 3.12.8, Platform: Windows-11-10.0.26100-SP0
2025-07-17 14:05:09,102 - INFO - Connecting to GLOBAL Snowflake domain
2025-07-17 14:05:09,384 - INFO - Kết nối Snowflake thành công!


=== QUYỀN CỦA ROLE SYSADMIN ===
- (datetime.datetime(2025, 7, 13, 20, 30, 14, 552000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'CREATE DATABASE', 'ACCOUNT', 'IJ46654', 'ROLE', 'SYSADMIN', 'true', '')
- (datetime.datetime(2025, 7, 13, 20, 30, 14, 553000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'CREATE WAREHOUSE', 'ACCOUNT', 'IJ46654', 'ROLE', 'SYSADMIN', 'true', '')
- (datetime.datetime(2025, 7, 15, 0, 30, 22, 357000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'CREATE SCHEMA', 'DATABASE', 'PERSON_TRANS_COUNTRY_DTB', 'ROLE', 'SYSADMIN', 'false', 'ACCOUNTADMIN')
- (datetime.datetime(2025, 7, 15, 0, 30, 22, 18000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'USAGE', 'DATABASE', 'PERSON_TRANS_COUNTRY_DTB', 'ROLE', 'SYSADMIN', 'false', 'ACCOUNTADMIN')
- (datetime.datetime(2025, 7, 15, 0, 30, 23, 80000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'CREATE TABLE', 'SCHEMA',

True

# Import List table - schema into snowflake

In [4]:
import pandas as pd
import pyodbc
import snowflake.connector
from pathlib import Path
import os

# ====== CONFIG ======
sql_server_config = {
    "server": "DD-HIEUHC",
    "database": "BicycleRetailer",
    "schema": "Sales"
}

snowflake_config = {
    "user": "Huynhhieu1710",
    "password": "17102003Huynhconghieu@",
    "account": "ij46654.southeast-asia.azure",
    "warehouse": "DATA_WAREHOUSE_DEMO",
    "database": "PERSON_TRANS_COUNTRY_DTB",
    "schema": "SALES_TEMP",
    "role": "ACCOUNTADMIN"
}

table_mappings = {
     "SalesReason": "SalesReason",
     "CreditCard": "CreditCard",
     "SalesTerritory": "SalesTerritory",
     "SalesPerson" : "SalesPerson",
     "SalesOrderHeader" : "SalesOrderHeader",
     "SalesOrderDetail" : "SalesOrderDetail",
     "Customer" : "Customer",
     "PersonCreditCard" : "PersonCreditCard",
     "CurrencyRate" : "CurrencyRate",
     "Currency" : "Currency",
     "SalesOrderHeaderSalesReason": "SalesOrderHeaderSalesReason",
     "Store" : "Store",
     "SpecialOffer" : "SpecialOffer"
   }


# table_mappings = {
#       "Product": "Product",
#       "ProductSubcategory": "ProductSubcategory",
#       "ProductCategory"  : "ProductCategory"
#   }

# ====== CONNECT SQL SERVER ======
def get_sql_server_connection():
    conn_str = (
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={sql_server_config['server']};"
        f"DATABASE={sql_server_config['database']};"
        f"Trusted_Connection=yes;"
        f"TrustServerCertificate=yes;"
    )
    return pyodbc.connect(conn_str)

# ====== CONNECT SNOWFLAKE ======
def connect_snowflake():
    return snowflake.connector.connect(**snowflake_config)

# ====== TẠO TABLE DỰA TRÊN DATAFRAME ======
def generate_create_table_sql(df, table_name):
    dtype_mapping = {
        'object': 'VARCHAR(50)',
        'int64': 'INTEGER',
        'float64': 'NUMBER(10,2)',
        'bool': 'BOOLEAN',
        'datetime64[ns]': 'TIMESTAMP_NTZ(3)'
    }

    # Danh sách các từ khóa cần escape
    reserved_keywords = {'group', 'order', 'user', 'rank', 'limit', 'date'}

    columns = []

    for col, dtype in df.dtypes.items():
        # Tối ưu VARCHAR cho object
        if pd.api.types.is_object_dtype(dtype):
            max_len = df[col].dropna().astype(str).map(len).max()
            max_len = max(1, min(max_len, 10000))
            sf_type = f'VARCHAR({max_len})'
        elif pd.api.types.is_integer_dtype(dtype):
            sf_type = 'NUMBER(7, 0)'
        elif pd.api.types.is_float_dtype(dtype):
            sf_type = 'NUMBER(10,2)'
        elif pd.api.types.is_bool_dtype(dtype):
            sf_type = 'BOOLEAN'
        elif pd.api.types.is_datetime64_any_dtype(dtype):
            sf_type = 'TIMESTAMP_NTZ(3)'
        else:
            sf_type = 'VARCHAR(100)'

        # Escape tên cột nếu là từ khóa
        col_escaped = f'"{col}"' if col.lower() in reserved_keywords else col

        columns.append(f'{col_escaped} {sf_type}')

    create_sql = f'CREATE OR REPLACE TABLE {table_name} (\n  {",\n  ".join(columns)}\n)'
    return create_sql

# ====== EXPORT TO CSV ======
def export_df_to_csv(df, filename):
    df.to_csv(filename, index=False)
    return Path(filename).resolve()

# ====== UPLOAD CSV + COPY INTO SNOWFLAKE ======
def upload_csv_to_snowflake(cursor, table_name, csv_path):
    put_command = f"PUT 'file://{csv_path.as_posix()}' @%{table_name} OVERWRITE = TRUE"
    cursor.execute(put_command)

    copy_command = f"""
        COPY INTO {table_name}
        FROM @%{table_name}
        FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1)
    """
    cursor.execute(copy_command)

# ====== MAIN PROCESS ======
def sync_all_tables():
    sql_conn = get_sql_server_connection()
    sf_conn = connect_snowflake()
    sf_cursor = sf_conn.cursor()

    for sql_table, sf_table in table_mappings.items():
        print(f"Đang xử lý bảng: {sql_table} → {sf_table}")

        # Lấy dữ liệu từ SQL Server
        df = pd.read_sql_query(f"SELECT * FROM {sql_server_config['schema']}.{sql_table}", sql_conn)

        # Tạo bảng trên Snowflake
        create_sql = generate_create_table_sql(df, sf_table)
        sf_cursor.execute(create_sql)

        # Export CSV
        csv_filename = f"{sf_table}.csv"
        csv_path = export_df_to_csv(df, csv_filename)

        # Upload + COPY
        upload_csv_to_snowflake(sf_cursor, sf_table, csv_path)

        # Xóa file tạm
        os.remove(csv_path)

        print(f"Hoàn tất bảng {sf_table}\n")

        print("Hoàn tất import toàn bộ bảng!")

# Gọi hàm chính
sync_all_tables()

Đang xử lý bảng: SalesReason → SalesReason


C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel_20380\3258247553.py:129: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {sql_server_config['schema']}.{sql_table}", sql_conn)


Hoàn tất bảng SalesReason

Hoàn tất import toàn bộ bảng!
Đang xử lý bảng: CreditCard → CreditCard


C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel_20380\3258247553.py:129: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {sql_server_config['schema']}.{sql_table}", sql_conn)


Hoàn tất bảng CreditCard

Hoàn tất import toàn bộ bảng!
Đang xử lý bảng: SalesTerritory → SalesTerritory


...

Hoàn tất bảng SalesOrderHeaderSalesReason

Hoàn tất import toàn bộ bảng!
Đang xử lý bảng: Store → Store


C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel_20380\3258247553.py:129: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {sql_server_config['schema']}.{sql_table}", sql_conn)


Hoàn tất bảng Store

Hoàn tất import toàn bộ bảng!
Đang xử lý bảng: SpecialOffer → SpecialOffer


C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel_20380\3258247553.py:129: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {sql_server_config['schema']}.{sql_table}", sql_conn)


Hoàn tất bảng SpecialOffer

Hoàn tất import toàn bộ bảng!


## Connect to Snowflake

In [1]:
import pandas as pd
import pyodbc
import snowflake.connector
from pathlib import Path
import os

snowflake_config = {
    "user": "Huynhhieu1710",
    "password": "17102003Huynhconghieu@",
    "account": "ij46654.southeast-asia.azure",
    "warehouse": "DATA_WAREHOUSE_DEMO",
    "database": "PERSON_TRANS_COUNTRY_DTB",
    "schema": "SALES_TEMP",
    "role": "ACCOUNTADMIN"
}

def connect_snowflake():
    return snowflake.connector.connect(**snowflake_config)

In [7]:
from pathlib import Path
import os

# === 2. KẾT NỐI SNOWFLAKE ===
sf_conn = snowflake.connector.connect(
    user='Huynhhieu1710',
    password='17102003Huynhconghieu@',
    account='ij46654.southeast-asia.azure',
    warehouse='DATA_WAREHOUSE_DEMO',
    database='PERSON_TRANS_COUNTRY_DTB',
    schema='SALES_PRODUCT_DWH',
    role='ACCOUNTADMIN'
)
cursor = sf_conn.cursor()

#### Export to csv

In [39]:
query_1 = """SELECT * FROM SALES_PRODUCT_DWH.DIMCREDITCARD"""
df_1 = pd.read_sql(query_1, sf_conn)

query_2 = """SELECT * FROM SALES_PRODUCT_DWH.DIMCURRENCY"""
df_2 = pd.read_sql(query_2, sf_conn)

query_3 = """SELECT * FROM SALES_PRODUCT_DWH.DIMCUSTOMER"""
df_3 = pd.read_sql(query_3, sf_conn)

query_4 = """SELECT * FROM SALES_PRODUCT_DWH.DIMDATE"""
df_4 = pd.read_sql(query_4, sf_conn)

query_5 = """SELECT * FROM SALES_PRODUCT_DWH.DIMPRODUCTCOSTHISTORY"""
df_5 = pd.read_sql(query_5, sf_conn)

query_6 = """SELECT * FROM SALES_PRODUCT_DWH.DIMPRODUCTINVENTORY"""
df_6 = pd.read_sql(query_6, sf_conn)

query_7 = """SELECT * FROM SALES_PRODUCT_DWH.DIMSALESPERSON"""
df_7 = pd.read_sql(query_7, sf_conn)

query_8 = """SELECT * FROM SALES_PRODUCT_DWH.DIMSHIPMETHOD"""
df_8 = pd.read_sql(query_8, sf_conn)

query_9 = """SELECT * FROM SALES_PRODUCT_DWH.DIMSPECIALOFFER"""
df_9 = pd.read_sql(query_9, sf_conn)

query_10 = """SELECT * FROM SALES_PRODUCT_DWH.DIMTERRITORY"""
df_10 = pd.read_sql(query_10, sf_conn)

query_11 = """SELECT * FROM SALES_PRODUCT_DWH.DIMWORKORDERROUTING"""
df_11 = pd.read_sql(query_11, sf_conn)

query_12 = """SELECT * FROM SALES_PRODUCT_DWH.FACTPRODUCT"""
df_12 = pd.read_sql(query_12, sf_conn)

query_13 = """SELECT * FROM SALES_PRODUCT_DWH.FACTPURCHASEORDERDETAIL"""
df_13 = pd.read_sql(query_13, sf_conn)

query_14 = """SELECT * FROM SALES_PRODUCT_DWH.FACTPURCHASEORDERHEADER"""
df_14 = pd.read_sql(query_14, sf_conn)

query_15 = """SELECT * FROM SALES_PRODUCT_DWH.FACTSALESORDERDETAIL"""
df_15 = pd.read_sql(query_15, sf_conn)

query_16 = """SELECT * FROM SALES_PRODUCT_DWH.FACTSALESORDERHEADER"""
df_16 = pd.read_sql(query_16, sf_conn)

query_17 = """SELECT * FROM SALES_PRODUCT_DWH.FACTSALESORDERREASON"""
df_17 = pd.read_sql(query_17, sf_conn)

query_18 = """SELECT * FROM SALES_PRODUCT_DWH.DIMSTORE"""
df_18 = pd.read_sql(query_18, sf_conn)

C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel_16816\2341987821.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_1 = pd.read_sql(query_1, sf_conn)
C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel_16816\2341987821.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_2 = pd.read_sql(query_2, sf_conn)
C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel_16816\2341987821.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_3 = pd.read_sql(query_3, sf_conn)
C:\Users\hieu.huynhcong\AppData\Local\Temp\ipykernel

In [ ]:
import pandas as pd

# Tạo dict chứa tên bảng và DataFrame tương ứng
dfs = {
    # "DIMCREDITCARD": df_1,
    # "DIMCURRENCY": df_2,
    # "DIMCUSTOMER": df_3,
    # "DIMDATE": df_4,
    # "DIMPRODUCTCOSTHISTORY": df_5,
    # "DIMPRODUCTINVENTORY": df_6,
    # "DIMSALESPERSON": df_7,
    # "DimCustomer": df_3
    # "DIMSPECIALOFFER": df_9,
    # "DIMTERRITORY": df_10,
    # "DIMWORKORDERROUTING": df_11,
    # "FactSalesOrderDetail": df_15,
    # "FactSalesOrderHeader": df_16
    # "FactPurchaseOrderDetail" : df_13, 
    # "DimSalesStore" : df_18
    # "FACTSALESORDERREASON": df_17
    "DimCustomer" : df_3
}

# Xuất từng DataFrame ra CSV
for table_name, df in dfs.items():
    file_name = f"{table_name}.csv"
    df.to_csv(file_name, index=False, encoding="utf-8-sig")
    print(f"Đã xuất {file_name}")

Đã xuất DimCustomer.csv
